# WikiPulse — Phase 3: Data Cleaning for Bot Detection

**Goal:** load raw edits from the Bronze Delta table, drop columns that carry
no useful signal, handle structural issues (duplicates, nulls, anonymous
editors), and get to a clean per-edit dataframe we can build per-user
behavioral features on top of.

**Important:** this notebook reads Bronze but does not modify it. Bronze
stays raw and untouched — all cleaning decisions here only affect what *this
analysis* works with, not the underlying source of truth.

## Step 1: Load Bronze

In [ ]:
# Lightweight Delta reader — no Spark session needed for exploratory work
# pip install deltalake pandas s3fs pyarrow

import pandas as pd
from deltalake import DeltaTable

storage_options = {
    "AWS_ENDPOINT_URL": "http://minio:9000",
    "AWS_ACCESS_KEY_ID": "wikipulse",
    "AWS_SECRET_ACCESS_KEY": "wikipulse123",
    "AWS_REGION": "us-east-1",       # required by the S3 client even though MinIO ignores it
    "AWS_ALLOW_HTTP": "true",        # MinIO isn't using HTTPS locally
}

dt = DeltaTable("s3a://wikipulse-bronze/edits", storage_options=storage_options)
df = dt.to_pandas()

print(f"Loaded {len(df):,} rows")
df.dtypes

## Step 2: Drop noise columns

Reasoning (see conversation with Claude for full detail):

| Column | Why it's dropped here |
|---|---|
| `server_url`, `server_name`, `server_script_path` | Constant for every row while we're only ingesting `enwiki` — zero information content |
| `parsedcomment` | HTML-rendered duplicate of `comment` — adds nothing for behavioral features |

`wiki` is **kept** even though it's currently constant too — the moment the
producer's `WIKI_FILTER` widens beyond English Wikipedia, this column stops
being noise and becomes essential. No cost to keeping it now.

In [ ]:
COLUMNS_TO_DROP = ["server_url", "server_name", "server_script_path", "parsedcomment"]

df_clean = df.drop(columns=COLUMNS_TO_DROP)
df_clean.columns.tolist()

## Step 3: Structural cleaning

- **Deduplicate** on the edit's own revision `id` — Structured Streaming with
  `foreachBatch` can occasionally write the same event twice if a micro-batch
  is interrupted mid-write and retried.
- **Null `comment` handling** — an empty edit summary is normal, not
  malformed. Fill with `""` rather than dropping the row.
- **Flag anonymous editors** — `user` is sometimes an IP address rather than
  a username (anonymous edit). Don't drop these; flag them, since they're a
  separate meaningful signal later, not something invalid.

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["id"], keep="first")
print(f"Dropped {before - len(df_clean):,} duplicate row(s) on revision id")

In [ ]:
df_clean["comment"] = df_clean["comment"].fillna("")
df_clean["parsedcomment"] = None  # column already dropped above; harmless if re-run
print("Null comments filled")

In [ ]:
import re

# Matches IPv4 and basic IPv6 — good enough to distinguish "this is an IP,
# not a registered username" without needing a full IP-validation library.
IP_PATTERN = re.compile(r"^(\d{1,3}\.){3}\d{1,3}$|^[0-9a-fA-F:]+$")

df_clean["is_anonymous"] = df_clean["user"].apply(lambda u: bool(IP_PATTERN.match(str(u))))

print(f"Anonymous edits: {df_clean['is_anonymous'].sum():,} of {len(df_clean):,} "
      f"({df_clean['is_anonymous'].mean():.1%})")

## Step 4: Sanity check the cleaned data

In [ ]:
print(df_clean.shape)
df_clean.head()

In [ ]:
# Quick check: does anything look structurally wrong that we haven't handled?
print("Nulls per column:")
print(df_clean.isnull().sum())

print("\nBot vs non-bot edit counts:")
print(df_clean["bot"].value_counts())

## Next

With `df_clean` in hand, the next step is building **per-user behavioral
features** (edit frequency, timing regularity, edit size patterns, comment
habits) aggregated across all of a user's edits — that's what the bot
classifier will actually train on, not individual edit rows.